<h2>Description</h2>

L'objectif de ce code est de fusionner toutes les données, donc à savoir les données de la rue des saints-peres avec les données externes. 
Nous voulons donc créer un dataframe qui contient :
<li>les informations météorologiques</li>
<li>les informations sur les vacances et jour fériés</li>
<li>les informations sur l'opération "Paris respire"</li>

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

doc = 'sts_peres.csv'

print("Version installée de pandas : ")
print(pd.__version__)

Version installée de pandas : 
2.3.0


<h3>Analyse du dataset relatif à l'axe</h3>

In [2]:
df_axe = pd.read_csv("../datasets_axes_bruts/" + doc, sep=";")

# On convertit la date en format convenable 
df_axe['Date et heure de comptage'] = pd.to_datetime(df_axe['Date et heure de comptage'], 
                                      errors='coerce', utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)  

df_axe.head() 

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,Etat arc,Date debut dispo data,Date fin dispo data,geo_point_2d,geo_shape
0,191,Sts_Peres,2025-04-01 10:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,Invalide,1996-10-03,2023-01-01,"48.85728037029827, 2.332454190717672","{""coordinates"": [[2.3332577811223216, 48.85826..."
1,191,Sts_Peres,2025-06-04 23:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,Invalide,1996-10-03,2023-01-01,"48.85728037029827, 2.332454190717672","{""coordinates"": [[2.3332577811223216, 48.85826..."
2,191,Sts_Peres,2025-06-04 22:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,Invalide,1996-10-03,2023-01-01,"48.85728037029827, 2.332454190717672","{""coordinates"": [[2.3332577811223216, 48.85826..."
3,191,Sts_Peres,2025-06-04 20:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,Invalide,1996-10-03,2023-01-01,"48.85728037029827, 2.332454190717672","{""coordinates"": [[2.3332577811223216, 48.85826..."
4,191,Sts_Peres,2025-06-04 19:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,Invalide,1996-10-03,2023-01-01,"48.85728037029827, 2.332454190717672","{""coordinates"": [[2.3332577811223216, 48.85826..."


In [3]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval
count,8747.0,8747,1404.000000,1404.000000,8747.0,8747.0
mean,191.0,2025-04-26 02:38:19.028238336,469.792023,7.137703,114.0,119.0
min,191.0,2024-10-01 05:00:00,34.000000,0.242220,114.0,119.0
25%,191.0,2025-01-22 16:30:00,265.500000,3.421945,114.0,119.0
50%,191.0,2025-04-25 20:00:00,524.000000,6.955280,114.0,119.0
75%,191.0,2025-08-06 21:30:00,651.000000,10.014165,114.0,119.0
max,191.0,2025-11-07 00:00:00,1083.000000,26.115560,114.0,119.0
std,0.0,NaN,225.105958,4.280484,0.0,0.0


In [4]:
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              1404 non-null   float64       
 4   Taux d'occupation          1404 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [5]:
df_axe['date'] = df_axe['Date et heure de comptage'].dt.date
df_axe['heure'] = df_axe['Date et heure de comptage'].dt.hour
df_axe['dow'] = df_axe['Date et heure de comptage'].dt.dayofweek
df_axe['mois'] = df_axe['Date et heure de comptage'].dt.month
df_axe['annee'] = df_axe['Date et heure de comptage'].dt.year
df_axe['jour_mois'] = df_axe['Date et heure de comptage'].dt.day
jour_map = {0:'Lun',1:'Mar',2:'Mer',3:'Jeu',4:'Ven',5:'Sam',6:'Dim'}
df_axe['jour_semaine'] = df_axe['dow'].map(jour_map)

In [6]:
fig = px.line(
    df_axe.sort_values('Date et heure de comptage'),
    x='Date et heure de comptage', y='Débit horaire',
    title=f"Débit horaire",
    labels={'Date et heure de comptage':'Date/Heure','Débit horaire':'Débit (véh/h)'}
)

fig.show()

In [7]:
profil = (df_axe
          .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
          .mean())

fig = px.line(
    profil, x='heure', y='Débit horaire', color='jour_semaine',
    markers=True, title=f"Profil horaire moyen par jour",
    labels={'heure':'Heure','Débit horaire':'Débit moyen (véh/h)','jour_semaine':'Jour'}
)
fig.update_layout(template='simple_white')
fig.show()

In [8]:
heat = (df_axe
        .groupby(['jour_semaine','heure'], as_index=False)['Débit horaire']
        .mean())


heat['jour_semaine'] = pd.Categorical(heat['jour_semaine'],
                                      categories=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
                                      ordered=True)

fig = px.imshow(
    heat.pivot(index='jour_semaine', columns='heure', values='Débit horaire').values,
    labels=dict(x="Heure", y="Jour", color="Débit moyen"),
    x=list(range(24)),
    y=['Lun','Mar','Mer','Jeu','Ven','Sam','Dim'],
    title=f"Carte de chaleur"
)
fig.update_layout(template='simple_white')
fig.show()


In [9]:
df_scatter = df_axe.dropna(subset=['Débit horaire','Taux d\'occupation']).copy()
fig = px.scatter(
    df_scatter, x='Taux d\'occupation', y='Débit horaire',
    color='jour_semaine', opacity=0.7,
    title=f"Débit vs Taux d’occupation",
    labels={'Taux d\'occupation':'Taux d’occupation','Débit horaire':'Débit (véh/h)'}
)

fig.update_layout(template='simple_white')
fig.show()

In [10]:
box = df_axe.dropna(subset=['Débit horaire']).copy()
fig = px.box(
    box, x='jour_semaine', y='Débit horaire', points='all',
    title=f"Distribution du débit par jour",
    labels={'jour_semaine':'Jour','Débit horaire':'Débit (véh/h)'}
)
fig.update_layout(template='simple_white')
fig.show()


In [11]:
etat_jour = (df_axe
             .assign(jour=pd.to_datetime(df_axe['date']))
             .groupby(['jour','Etat trafic'], as_index=False)
             .size())

fig = px.area(
    etat_jour, x='jour', y='size', color='Etat trafic',
    title=f"État du trafic (comptes journaliers)",
    labels={'jour':'Date','size':'Occurrences'}
)
fig.update_layout(template='simple_white', hovermode='x unified')
fig.show()

In [12]:
mensuel = (df_axe
           .set_index('Date et heure de comptage')
           .resample('MS')['Débit horaire']
           .mean()
           .to_frame('debit_moyen')
           .reset_index())

mensuel['trend_3m'] = mensuel['debit_moyen'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['debit_moyen'],
    mode='lines+markers', name='Moyenne mensuelle'
))
fig.add_trace(go.Scatter(
    x=mensuel['Date et heure de comptage'], y=mensuel['trend_3m'],
    mode='lines', name='Tendance (MM-3)', line=dict(width=4)
))

fig.update_layout(
    title="Débit horaire – tendance mensuelle (moyenne + lissage 3 mois)",
    xaxis_title="Mois",
    yaxis_title="Débit moyen (véh/h)",
    template="simple_white",
    hovermode="x unified"
)
fig.show()


In [13]:
profil = (df_axe
          .groupby(['annee','mois'], as_index=False)['Débit horaire']
          .mean()
          .rename(columns={'Débit horaire':'debit_moyen'}))

fig = px.line(
    profil, x='mois', y='debit_moyen', color='annee',
    markers=True,
    title="Profil mensuel du débit par année",
    labels={'mois':'Mois', 'debit_moyen':'Débit moyen (véh/h)', 'annee':'Année'}
)
fig.update_layout(template='simple_white', xaxis=dict(dtick=1))
fig.show()


In [14]:
df_axe['heure_sin'] = np.sin(2 * np.pi * df_axe['heure'] / 24)
df_axe['heure_cos'] = np.cos(2 * np.pi * df_axe['heure'] / 24)

df_axe['jour_sin'] = np.sin(2 * np.pi * df_axe['dow'] / 7)
df_axe['jour_cos'] = np.cos(2 * np.pi * df_axe['dow'] / 7)

df_axe['mois_sin'] = np.sin(2 * np.pi * df_axe['mois'] / 12)
df_axe['mois_cos'] = np.cos(2 * np.pi * df_axe['mois'] / 12)

In [15]:
df_axe.head(10)

,Identifiant arc,Libelle,Date et heure de comptage,Débit horaire,Taux d'occupation,Etat trafic,Identifiant noeud amont,Libelle noeud amont,Identifiant noeud aval,Libelle noeud aval,...,mois,annee,jour_mois,jour_semaine,heure_sin,heure_cos,jour_sin,jour_cos,mois_sin,mois_cos
0,191,Sts_Peres,2025-04-01 10:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,4,2025,1,Mar,5.000000e-01,-8.660254e-01,0.781831,0.623490,8.660254e-01,-0.5
1,191,Sts_Peres,2025-06-04 23:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-2.588190e-01,9.659258e-01,0.974928,-0.222521,1.224647e-16,-1.0
2,191,Sts_Peres,2025-06-04 22:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-5.000000e-01,8.660254e-01,0.974928,-0.222521,1.224647e-16,-1.0
3,191,Sts_Peres,2025-06-04 20:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-8.660254e-01,5.000000e-01,0.974928,-0.222521,1.224647e-16,-1.0
4,191,Sts_Peres,2025-06-04 19:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-9.659258e-01,2.588190e-01,0.974928,-0.222521,1.224647e-16,-1.0
5,191,Sts_Peres,2025-06-04 18:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-1.000000e+00,-1.836970e-16,0.974928,-0.222521,1.224647e-16,-1.0
6,191,Sts_Peres,2025-06-04 17:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,6,2025,4,Mer,-9.659258e-01,-2.588190e-01,0.974928,-0.222521,1.224647e-16,-1.0
7,191,Sts_Peres,2025-10-09 12:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,10,2025,9,Jeu,1.224647e-16,-1.000000e+00,0.433884,-0.900969,-8.660254e-01,0.5
8,191,Sts_Peres,2025-10-09 13:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,10,2025,9,Jeu,-2.588190e-01,-9.659258e-01,0.433884,-0.900969,-8.660254e-01,0.5
9,191,Sts_Peres,2025-10-09 14:00:00,NaN,NaN,Inconnu,114,Sts_Peres-Voltaire,119,Sts_Peres-Universite,...,10,2025,9,Jeu,-5.000000e-01,-8.660254e-01,0.433884,-0.900969,-8.660254e-01,0.5


<h3>Nous insérons d'abord les données météorologiques</h3>

In [16]:
meteo = pd.read_csv("../datasets_externes_clean/meteo.csv", sep=";")
meteo['datetime'] = pd.to_datetime(meteo['datetime'])

In [17]:
df_merge = pd.merge(df_axe, meteo, left_on="Date et heure de comptage", right_on="datetime", how="left")

In [18]:
df_merge.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,jour_cos,mois_sin,mois_cos,Unnamed: 0,precipitations heure,duree prec (en min),force moyenne vent (m/s),Température,ensoleillement (en min),datetime
count,8747.0,8747,1404.000000,1404.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8.747000e+03,8.747000e+03,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747
mean,191.0,2025-04-26 02:38:19.028238336,469.792023,7.137703,114.0,119.0,11.506574,2.994741,6.668000,2024.804047,...,-0.006239,-5.500528e-02,3.106604e-02,11546.638619,0.077867,3.980107,2.933686,13.218383,13.081685,2025-04-26 02:38:19.028238336
min,191.0,2024-10-01 05:00:00,34.000000,0.242220,114.0,119.0,0.000000,0.000000,1.000000,2024.000000,...,-0.900969,-1.000000e+00,-1.000000e+00,6581.000000,0.000000,0.000000,0.000000,-3.600000,0.000000,2024-10-01 05:00:00
25%,191.0,2025-01-22 16:30:00,265.500000,3.421945,114.0,119.0,6.000000,1.000000,4.000000,2025.000000,...,-0.900969,-8.660254e-01,-5.000000e-01,9304.500000,0.000000,0.000000,2.000000,8.700000,0.000000,2025-01-22 16:30:00
50%,191.0,2025-04-25 20:00:00,524.000000,6.955280,114.0,119.0,12.000000,3.000000,7.000000,2025.000000,...,-0.222521,-2.449294e-16,6.123234e-17,11540.000000,0.000000,0.000000,2.800000,13.000000,0.000000,2025-04-25 20:00:00
75%,191.0,2025-08-06 21:30:00,651.000000,10.014165,114.0,119.0,17.000000,5.000000,10.000000,2025.000000,...,0.623490,5.000000e-01,5.000000e-01,14013.500000,0.000000,0.000000,3.700000,17.700000,20.000000,2025-08-06 21:30:00
max,191.0,2025-11-07 00:00:00,1083.000000,26.115560,114.0,119.0,23.000000,6.000000,12.000000,2025.000000,...,1.000000,1.000000e+00,1.000000e+00,16224.000000,14.300000,60.000000,9.300000,37.600000,60.000000,2025-11-07 00:00:00
std,0.0,NaN,225.105958,4.280484,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,0.707910,7.313334e-01,6.791720e-01,2769.396136,0.518787,12.900969,1.311747,6.678668,21.956183,NaN


In [19]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 36 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              1404 non-null   float64       
 4   Taux d'occupation          1404 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date f

In [20]:
df_axe = df_merge

<h3>Nous allons maintenant ajouter les données relatives au jours piétonnisés</h3>

Nous allons donc ajouter une colonne de 1 ou 0 pour indiquer si oui ou non il s'agissait d'un jour piéton.

In [21]:
pieton = pd.read_csv("../datasets_externes_clean/pieton.csv", sep=";")
pieton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  20 non-null     int64 
 1   date_debut  20 non-null     object
 2   date_fin    20 non-null     object
dtypes: int64(1), object(2)
memory usage: 608.0+ bytes


In [22]:
pieton['date_debut'] = pd.to_datetime(pieton['date_debut'])
pieton['date_fin'] = pd.to_datetime(pieton['date_fin'])

Nous écrivons ci-dessous une fonction pour déterminer si pour une date donnée, certains axes parisiens étaient piétonnisés.

In [23]:
def est_pietonnise(date):
    debut = pieton['date_debut']
    fin = pieton['date_fin']
    for i in range (len(debut)):
        if date >= debut[i] and date < fin[i]:
            return 1
    return 0

df_axe['est_pieton'] = df_axe['Date et heure de comptage'].apply(est_pietonnise)

In [24]:
df_axe[df_axe['est_pieton'] == 1][['Date et heure de comptage', 'est_pieton']]

,Date et heure de comptage,est_pieton
159,2024-11-03 16:00:00,1
160,2024-11-03 15:00:00,1
161,2024-11-03 12:00:00,1
268,2024-11-03 14:00:00,1
269,2024-11-03 13:00:00,1
...,...,...
6902,2025-09-21 17:00:00,1
6903,2025-09-21 13:00:00,1
6904,2025-09-21 10:00:00,1
6905,2025-09-21 09:00:00,1


In [25]:
df_axe = df_axe.sort_values(by="Date et heure de comptage", ascending=True)

<h3>On va maintenant ajouter des données sur les vacances, jours fériés...</h3>

In [26]:
vacances = pd.read_csv("../datasets_externes_clean/vacances.csv", sep=";")
vacances ['Date de début'] = pd.to_datetime(vacances['Date de début'])
vacances ['Date de fin'] = pd.to_datetime(vacances['Date de fin'])

In [27]:
def est_jour_vacances(dat):
    date_sans_heure = dat.date()
    debut = vacances['Date de début'].dt.date
    fin = vacances['Date de fin'].dt.date
    for i in range (len(debut)):
        if date_sans_heure >= debut[i] and date_sans_heure < fin[i]:
            return 1
    return 0

df_axe['est_vacances'] = df_axe['Date et heure de comptage'].apply(est_jour_vacances)

In [28]:
df = df_axe[df_axe['est_vacances'] == 1][['Date et heure de comptage', 'est_vacances']]

In [29]:
df.head(50)

,Date et heure de comptage,est_vacances
5188,2024-10-19 00:00:00,1
4990,2024-10-19 01:00:00,1
4989,2024-10-19 02:00:00,1
5331,2024-10-19 03:00:00,1
5330,2024-10-19 04:00:00,1
4988,2024-10-19 05:00:00,1
5329,2024-10-19 06:00:00,1
5328,2024-10-19 07:00:00,1
4987,2024-10-19 08:00:00,1
5327,2024-10-19 09:00:00,1


In [30]:
def est_avant_vacances(date):
    date_sans_heure = date.date()
    debut = vacances['Date de début'].dt.date
    for i in range (len(debut)):
        if date_sans_heure == debut[i] - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_vacances'] = df_axe['Date et heure de comptage'].apply(est_avant_vacances)

In [31]:
df_axe[df_axe['est_avant_vacances'] == 1][['est_avant_vacances', 'Date et heure de comptage']]

,est_avant_vacances,Date et heure de comptage
6505,1,2024-10-18 00:00:00
6367,1,2024-10-18 01:00:00
6366,1,2024-10-18 02:00:00
6365,1,2024-10-18 03:00:00
6364,1,2024-10-18 04:00:00
...,...,...
4388,1,2025-10-17 19:00:00
4389,1,2025-10-17 20:00:00
4123,1,2025-10-17 21:00:00
4124,1,2025-10-17 22:00:00


In [32]:
ferie = pd.read_csv("../datasets_externes_clean/ferie.csv", sep=";")

ferie['Date de début'] = pd.to_datetime(ferie['Date de début'], format='%d/%m/%Y')

In [33]:
def est_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el:
            return 1
    return 0

df_axe['est_ferie'] = df_axe['Date et heure de comptage'].apply(est_ferie)

In [34]:
df_axe[df_axe['est_ferie']==1]['Date et heure de comptage']

8733   2024-11-01 00:00:00
8592   2024-11-01 01:00:00
7954   2024-11-01 02:00:00
7953   2024-11-01 03:00:00
3152   2024-11-01 04:00:00
               ...        
41     2025-11-01 19:00:00
42     2025-11-01 20:00:00
43     2025-11-01 21:00:00
365    2025-11-01 22:00:00
366    2025-11-01 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [35]:
def est_avant_ferie(date):
    date = date.date()
    j_ferie = ferie['Date de début'].dt.date
    for el in j_ferie :
        if date == el - pd.Timedelta(days=1):
            return 1
    return 0

df_axe['est_avant_ferie'] = df_axe['Date et heure de comptage'].apply(est_avant_ferie)

In [36]:
df_axe[df_axe['est_avant_ferie']==1]['Date et heure de comptage']

8632   2024-10-31 00:00:00
8726   2024-10-31 01:00:00
8725   2024-10-31 02:00:00
8746   2024-10-31 03:00:00
8724   2024-10-31 04:00:00
               ...        
8692   2025-10-31 19:00:00
8715   2025-10-31 20:00:00
8714   2025-10-31 21:00:00
8713   2025-10-31 22:00:00
8712   2025-10-31 23:00:00
Name: Date et heure de comptage, Length: 264, dtype: datetime64[ns]

In [37]:
df_axe['est_weekend'] = df_axe['dow'].isin([5, 6]).astype(int)

Nous vérifions maintenant la composition du dataset

In [38]:
df_axe = df_axe.drop('Unnamed: 0', axis = 1)
df_axe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8747 entries, 1232 to 2034
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Identifiant arc            8747 non-null   int64         
 1   Libelle                    8747 non-null   object        
 2   Date et heure de comptage  8747 non-null   datetime64[ns]
 3   Débit horaire              1404 non-null   float64       
 4   Taux d'occupation          1404 non-null   float64       
 5   Etat trafic                8747 non-null   object        
 6   Identifiant noeud amont    8747 non-null   int64         
 7   Libelle noeud amont        8747 non-null   object        
 8   Identifiant noeud aval     8747 non-null   int64         
 9   Libelle noeud aval         8747 non-null   object        
 10  Etat arc                   8747 non-null   object        
 11  Date debut dispo data      8747 non-null   object        
 12  Date fin

In [39]:
df_axe.describe()

,Identifiant arc,Date et heure de comptage,Débit horaire,Taux d'occupation,Identifiant noeud amont,Identifiant noeud aval,heure,dow,mois,annee,...,force moyenne vent (m/s),Température,ensoleillement (en min),datetime,est_pieton,est_vacances,est_avant_vacances,est_ferie,est_avant_ferie,est_weekend
count,8747.0,8747,1404.000000,1404.000000,8747.0,8747.0,8747.000000,8747.000000,8747.000000,8747.000000,...,8747.000000,8747.000000,8747.000000,8747,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000,8747.000000
mean,191.0,2025-04-26 02:38:19.028238336,469.792023,7.137703,114.0,119.0,11.506574,2.994741,6.668000,2024.804047,...,2.933686,13.218383,13.081685,2025-04-26 02:38:19.028238336,0.012804,0.356579,0.016577,0.030182,0.030182,0.282383
min,191.0,2024-10-01 05:00:00,34.000000,0.242220,114.0,119.0,0.000000,0.000000,1.000000,2024.000000,...,0.000000,-3.600000,0.000000,2024-10-01 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,191.0,2025-01-22 16:30:00,265.500000,3.421945,114.0,119.0,6.000000,1.000000,4.000000,2025.000000,...,2.000000,8.700000,0.000000,2025-01-22 16:30:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,191.0,2025-04-25 20:00:00,524.000000,6.955280,114.0,119.0,12.000000,3.000000,7.000000,2025.000000,...,2.800000,13.000000,0.000000,2025-04-25 20:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,191.0,2025-08-06 21:30:00,651.000000,10.014165,114.0,119.0,17.000000,5.000000,10.000000,2025.000000,...,3.700000,17.700000,20.000000,2025-08-06 21:30:00,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,191.0,2025-11-07 00:00:00,1083.000000,26.115560,114.0,119.0,23.000000,6.000000,12.000000,2025.000000,...,9.300000,37.600000,60.000000,2025-11-07 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,0.0,NaN,225.105958,4.280484,0.0,0.0,6.919751,1.993350,3.434265,0.396955,...,1.311747,6.678668,21.956183,NaN,0.112436,0.479016,0.127688,0.171097,0.171097,0.450184


In [40]:
df_axe.to_csv("../datasets_axes_with_all_features/" + doc , sep=";")